# SEN2SR Lite: Baseline vs. Fine-Tuned Ground-Truth Benchmark & Copernicus Evaluation

**Major Project — Spectral Enhancement**

### Dual-Section Evaluation Protocol:
1. **Section 1: Quantitative & Visual Benchmark on Held-Out Test Scenes (`test_scenes.json`)**
   - Strict 4-way evaluation against real NAIP aerial ground truth: `[10m Bicubic] | [Pretrained Baseline] | [Fine-Tuned (Ours)] | [Real NAIP]`.
   - Identical contrast stretch computed from the real aerial ground truth.
   - 3x zoom inspections of roads, field edges, and built-up structures.
   - Absolute error difference maps (`|Fine-Tuned - NAIP|` vs `|Baseline - NAIP|`).
   - Sanity metrics table: PSNR, SSIM, NDVI-MAE.
2. **Section 2: Full-Scene Copernicus Satellite Inference (Hann Window Tiling)**
   - Real-world deployment demonstration across Copernicus Sentinel-2 tiles.
   - Visual inspection caveat: No aerial ground truth exists for Copernicus scenes; edge metrics represent relative sharpness, not accuracy.


## Cell 1 — Mount Google Drive & Environment Setup


In [ ]:
import os, sys, glob, re, time, zipfile, warnings, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CPL_LOG_ERRORS'] = 'OFF'
os.environ['CPL_LOG'] = 'OFF'
os.environ['GDAL_PAM_ENABLED'] = 'NO'

warnings.filterwarnings('ignore')
for logger_name in ['rasterio', 'rasterio._env', 'rasterio.env', 'rasterio._base']:
    logging.getLogger(logger_name).setLevel(logging.ERROR)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJECT_DIR = '/content/drive/My Drive/Project_Folder'
DATASETS_DIR = os.path.join(PROJECT_DIR, 'Datasets')
RESULTS_DIR  = os.path.join(PROJECT_DIR, 'Results')
SPLIT_FILE   = os.path.join(RESULTS_DIR, 'test_scenes.json')

os.makedirs(RESULTS_DIR, exist_ok=True)

import torch
import torch.nn.functional as F
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__} | Active Device: {device} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')


## Cell 2 — Install Required Dependencies


In [ ]:
%pip install -q sen2sr mlstac rasterio safetensors pandas matplotlib scikit-image


## Cell 3 — Canonical Band Ordering & Radiometric Calibration Helpers


In [ ]:
import rasterio

# Canonical Band Ordering: [Red, Green, Blue, NIR] = [B04, B03, B02, B08]
SEN2NAIP_LR_ORDER = [0, 1, 2, 3]
SEN2NAIP_HR_ORDER = [0, 1, 2, 3]
WS_LR_ORDER       = [3, 2, 1, 7]
WS_HR_ORDER       = [0, 1, 2, 3]

SCALE_S2 = 10000.0
SCALE_NAIP = 255.0
SCALE_WS_HR = 4095.0

def read_raster(path):
    with rasterio.open(path) as src:
        arr = src.read()
        return arr.astype(np.float32)

def match_radiometry_patch(hr_patch, lr_patch):
    """
    Quantile-matches HR patch radiometry to LR patch radiometry per channel.
    Eliminates exposure/DN offsets between airborne camera and satellite reflectance.
    """
    matched = np.zeros_like(hr_patch)
    for c in range(hr_patch.shape[0]):
        src = hr_patch[c].ravel()
        tpl = lr_patch[c].ravel()
        s_quantiles = (np.argsort(np.argsort(src)) + 0.5) / len(src)
        matched[c] = np.quantile(tpl, s_quantiles).reshape(hr_patch[c].shape)
    return np.clip(matched, 0.0, 1.0)

def load_canonical_pair(lr_path, hr_path, is_worldstrat=False):
    lr_raw = read_raster(lr_path)
    hr_raw = read_raster(hr_path)
    if is_worldstrat:
        lr_4b = lr_raw[WS_LR_ORDER]
        hr_4b = hr_raw[WS_HR_ORDER]
        lr_scaled = np.clip(lr_4b, 0.0, 1.0)
        hr_scaled = np.clip(hr_4b / SCALE_WS_HR, 0.0, 1.0)
    else:
        lr_4b = lr_raw[SEN2NAIP_LR_ORDER]
        hr_4b = hr_raw[SEN2NAIP_HR_ORDER]
        lr_scaled = np.clip(lr_4b / SCALE_S2, 0.0, 1.0)
        hr_scaled = np.clip(hr_4b / SCALE_NAIP, 0.0, 1.0)

    assert lr_scaled.shape[0] == 4 and hr_scaled.shape[0] == 4
    return lr_scaled, hr_scaled

def shared_stretch(img_4b, ref_hr):
    """Shared percentile stretch computed strictly from the real NAIP ground truth RGB."""
    rgb = img_4b[[0, 1, 2]].transpose(1, 2, 0)
    ref_rgb = ref_hr[[0, 1, 2]].transpose(1, 2, 0)
    lo = np.percentile(ref_rgb, 2)
    hi = np.percentile(ref_rgb, 98)
    return np.clip((rgb - lo) / max(hi - lo, 1e-6), 0.0, 1.0)

def calc_ndvi(cube_4b):
    red, nir = cube_4b[0], cube_4b[3]
    return (nir - red) / (nir + red + 1e-6)


## Cell 4 — Load Both Pretrained Baseline & Fine-Tuned SEN2SR Lite Models


In [ ]:
import mlstac
import safetensors.torch
import urllib.request
from sen2sr.models.opensr_baseline.cnn import CNNSR

MODEL_DIR = '/content/model/SEN2SR_LITE'
os.makedirs(MODEL_DIR, exist_ok=True)
weights_path = os.path.join(MODEL_DIR, 'sr_model.safetensor')

if not os.path.exists(weights_path):
    print('Downloading official SEN2SR Lite model weights from Taco Foundation...')
    try:
        mlstac.download(
            file='https://huggingface.co/tacofoundation/SEN2SR/resolve/main/SEN2SRLite/main/mlm.json',
            output_dir=MODEL_DIR
        )
    except Exception as e:
        print(f'mlstac download notice ({e}); downloading safetensor directly...')
        u = 'https://huggingface.co/tacofoundation/SEN2SR/resolve/main/SEN2SRLite/main/sr_model.safetensor'
        urllib.request.urlretrieve(u, weights_path)

# 1. Pretrained Baseline Model
model_baseline = CNNSR(in_channels=4, out_channels=4, feature_channels=24, upscale=4, bias=True, train_mode=False, num_blocks=6)
base_weights = safetensors.torch.load_file(weights_path)
model_baseline.load_state_dict(base_weights)
model_baseline = model_baseline.to(device).eval()
print('Pretrained Baseline SEN2SR Lite loaded successfully.')

# 2. Fine-Tuned Model (Best Checkpoint)
model_finetuned = CNNSR(in_channels=4, out_channels=4, feature_channels=24, upscale=4, bias=True, train_mode=False, num_blocks=6)
BEST_CKPT = os.path.join(RESULTS_DIR, 'sen2sr_lite_compound_best.pth')
LAST_CKPT = os.path.join(RESULTS_DIR, 'sen2sr_lite_compound_last.pth')

if os.path.exists(BEST_CKPT):
    ckpt = torch.load(BEST_CKPT, map_location=device)
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model_finetuned.load_state_dict(state)
    print(f'Fine-Tuned SEN2SR Lite (Best Checkpoint) loaded from: {BEST_CKPT}')
elif os.path.exists(LAST_CKPT):
    ckpt = torch.load(LAST_CKPT, map_location=device)
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model_finetuned.load_state_dict(state)
    print(f'Fine-Tuned SEN2SR Lite (Last Checkpoint) loaded from: {LAST_CKPT}')
else:
    print('Notice: Fine-tuned checkpoint not found on disk. Initializing demonstration with baseline weights.')
    model_finetuned.load_state_dict(base_weights)

model_finetuned = model_finetuned.to(device).eval()

# Check weight delta norm
diff_norm = torch.norm(next(model_finetuned.parameters()) - next(model_baseline.parameters())).item()
print(f'Weight Difference Norm (Fine-Tuned vs Baseline): {diff_norm:.6f}')


## SECTION 1: Ground-Truth Benchmark on Held-Out Test Scenes

Evaluates $N=6$ held-out test scenes from `Results/test_scenes.json` side-by-side against real NAIP aerial ground truth.


In [ ]:
import json

assert os.path.exists(SPLIT_FILE), f"SPLIT_FILE not found: {SPLIT_FILE}! Run 04 training notebook first."

with open(SPLIT_FILE, 'r', encoding='utf-8') as f:
    split_meta = json.load(f)

test_scenes = split_meta.get('test_scenes', [])
assert len(test_scenes) >= 1, "test_scenes list in test_scenes.json is empty!"

N_EVAL = min(6, len(test_scenes))
eval_scenes = test_scenes[:N_EVAL]
print(f'Evaluating {N_EVAL} held-out test scenes against real NAIP aerial ground truth:')
for i, s in enumerate(eval_scenes):
    print(f"  [{i+1}/{N_EVAL}] Scene ID: {s['id']}")


### Cell 6 — N=6 Ground-Truth 4-Way Side-by-Side Comparison

Columns: `10m Input (Bicubic 4x) | Pretrained Baseline (2.5m) | Fine-Tuned (2.5m, Ours) | Real NAIP Ground Truth (VHR)`
**All four panels share the identical percentile stretch computed strictly from the real NAIP ground truth.**


In [ ]:
fig, axes = plt.subplots(N_EVAL, 4, figsize=(20, 5 * N_EVAL))
if N_EVAL == 1:
    axes = np.expand_dims(axes, 0)

test_eval_cache = []

for i, scene in enumerate(eval_scenes):
    lr_full, hr_full = load_canonical_pair(scene['lr'], scene['hr'], is_worldstrat=scene['is_ws'])
    cy, cx = lr_full.shape[1] // 2, lr_full.shape[2] // 2
    lr_crop = lr_full[:, cy-32:cy+32, cx-32:cx+32]
    hr_crop = hr_full[:, (cy-32)*4:(cy+32)*4, (cx-32)*4:(cx+32)*4]
    hr_matched = match_radiometry_patch(hr_crop, lr_crop)

    in_t = torch.from_numpy(lr_crop).unsqueeze(0).to(device)
    with torch.no_grad():
        sr_fine = model_finetuned(in_t).squeeze(0).cpu().numpy()
        sr_base = model_baseline(in_t).squeeze(0).cpu().numpy()
        bicubic = F.interpolate(in_t, scale_factor=4, mode='bicubic', align_corners=False).squeeze(0).cpu().numpy()

    test_eval_cache.append({
        'id': scene['id'],
        'lr_crop': lr_crop,
        'bicubic': bicubic,
        'sr_base': sr_base,
        'sr_fine': sr_fine,
        'hr_gt': hr_matched
    })

    # Render 4 panels
    axes[i, 0].imshow(shared_stretch(bicubic, hr_matched))
    axes[i, 0].set_title(f"1. Native 10m Input (Bicubic 4x)\nScene: {scene['id']}", fontsize=11, fontweight='bold')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(shared_stretch(sr_base, hr_matched))
    axes[i, 1].set_title(f"2. Pretrained Baseline (2.5m)\nOfficial Weights", fontsize=11, fontweight='bold')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(shared_stretch(sr_fine, hr_matched))
    axes[i, 2].set_title(f"3. Fine-Tuned (2.5m, Ours)\nShift-Tolerant Loss", fontsize=11, fontweight='bold', color='green')
    axes[i, 2].axis('off')

    axes[i, 3].imshow(shared_stretch(hr_matched, hr_matched))
    axes[i, 3].set_title(f"4. Real NAIP Aerial Ground Truth\nAerial Reference (VHR)", fontsize=11, fontweight='bold')
    axes[i, 3].axis('off')

plt.tight_layout()
preview_path = os.path.join(RESULTS_DIR, 'sen2sr_lite_groundtruth_6patch_comparison.png')
plt.savefig(preview_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'Ground-truth comparison figure saved to: {preview_path}')


### Cell 7 — 3× Magnified Structural Zoom-In Inspection (Roads, Field Edges, Built-Up Clusters)


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
zoom_labels = ['Road / Linear Boundary', 'Agricultural Field Edge', 'Built-Up Cluster / Structures']

# Select 3 samples or quadrants
for row in range(3):
    sample = test_eval_cache[row % len(test_eval_cache)]
    gt = sample['hr_gt']
    fine = sample['sr_fine']
    base = sample['sr_base']
    bic = sample['bicubic']

    # Crop center 80x80 patch for 3x zoom
    zy, zx = gt.shape[1] // 2, gt.shape[2] // 2
    r_gt = gt[:, zy-40:zy+40, zx-40:zx+40]
    r_fine = fine[:, zy-40:zy+40, zx-40:zx+40]
    r_base = base[:, zy-40:zy+40, zx-40:zx+40]
    r_bic = bic[:, zy-40:zy+40, zx-40:zx+40]

    axes[row, 0].imshow(shared_stretch(r_bic, r_gt))
    axes[row, 0].set_title(f"10m Bicubic 4x [3x Zoom]\n{zoom_labels[row]} ({sample['id']})", fontweight='bold')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(shared_stretch(r_base, r_gt))
    axes[row, 1].set_title(f"Pretrained Baseline [3x Zoom]\n{zoom_labels[row]}", fontweight='bold')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(shared_stretch(r_fine, r_gt))
    axes[row, 2].set_title(f"Fine-Tuned (Ours) [3x Zoom]\n{zoom_labels[row]}", fontweight='bold', color='green')
    axes[row, 2].axis('off')

    axes[row, 3].imshow(shared_stretch(r_gt, r_gt))
    axes[row, 3].set_title(f"Real Aerial NAIP [3x Zoom]\n{zoom_labels[row]}", fontweight='bold')
    axes[row, 3].axis('off')

plt.tight_layout()
zoom_path = os.path.join(RESULTS_DIR, 'sen2sr_lite_zoom_features.png')
plt.savefig(zoom_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'Zoom comparison saved to: {zoom_path}')


### Cell 8 — Reconstruction Error Maps against Real NAIP Aerial Ground Truth

Visually demonstrates spatial error reduction: `|Baseline - NAIP|` vs `|Fine-Tuned - NAIP|`.


In [ ]:
fig, axes = plt.subplots(N_EVAL, 2, figsize=(14, 4 * N_EVAL))
if N_EVAL == 1:
    axes = np.expand_dims(axes, 0)

for i in range(N_EVAL):
    sample = test_eval_cache[i]
    err_base = np.mean(np.abs(sample['sr_base'] - sample['hr_gt']), axis=0)
    err_fine = np.mean(np.abs(sample['sr_fine'] - sample['hr_gt']), axis=0)

    vmax = max(np.percentile(err_base, 98), np.percentile(err_fine, 98), 1e-3)

    im0 = axes[i, 0].imshow(err_base, cmap='magma', vmin=0.0, vmax=vmax)
    axes[i, 0].set_title(f"Baseline Error |Base - NAIP|\nMean: {err_base.mean():.4f} ({sample['id']})", fontweight='bold')
    axes[i, 0].axis('off')
    plt.colorbar(im0, ax=axes[i, 0], fraction=0.046, pad=0.04)

    im1 = axes[i, 1].imshow(err_fine, cmap='magma', vmin=0.0, vmax=vmax)
    axes[i, 1].set_title(f"Fine-Tuned Error |Fine - NAIP| (Ours)\nMean: {err_fine.mean():.4f} ({sample['id']})", fontweight='bold', color='green')
    axes[i, 1].axis('off')
    plt.colorbar(im1, ax=axes[i, 1], fraction=0.046, pad=0.04)

plt.tight_layout()
diff_path = os.path.join(RESULTS_DIR, 'sen2sr_lite_error_difference_maps.png')
plt.savefig(diff_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'Error maps saved to: {diff_path}')


### Cell 9 — Quantitative Benchmark Sanity Table against Real Ground Truth

Computes PSNR, SSIM, and NDVI-MAE across held-out test patches.


In [ ]:
rows = []
for sample in test_eval_cache:
    gt = sample['hr_gt']
    base = sample['sr_base']
    fine = sample['sr_fine']

    # PSNR & SSIM on RGB channels
    gt_rgb = np.clip(gt[[0, 1, 2]].transpose(1, 2, 0), 0.0, 1.0)
    base_rgb = np.clip(base[[0, 1, 2]].transpose(1, 2, 0), 0.0, 1.0)
    fine_rgb = np.clip(fine[[0, 1, 2]].transpose(1, 2, 0), 0.0, 1.0)

    p_base = psnr_fn(gt_rgb, base_rgb, data_range=1.0)
    p_fine = psnr_fn(gt_rgb, fine_rgb, data_range=1.0)

    s_base = ssim_fn(gt_rgb, base_rgb, channel_axis=2, data_range=1.0)
    s_fine = ssim_fn(gt_rgb, fine_rgb, channel_axis=2, data_range=1.0)

    # NDVI MAE
    ndvi_gt = calc_ndvi(gt)
    ndvi_base = calc_ndvi(base)
    ndvi_fine = calc_ndvi(fine)

    ndvi_mae_base = np.nanmean(np.abs(ndvi_base - ndvi_gt))
    ndvi_mae_fine = np.nanmean(np.abs(ndvi_fine - ndvi_gt))

    rows.append({
        'Scene ID': sample['id'],
        'Baseline PSNR (dB)': round(p_base, 2),
        'Fine-Tuned PSNR (dB)': round(p_fine, 2),
        'PSNR Delta': round(p_fine - p_base, 2),
        'Baseline SSIM': round(s_base, 4),
        'Fine-Tuned SSIM': round(s_fine, 4),
        'SSIM Delta': round(s_fine - s_base, 4),
        'Baseline NDVI-MAE': round(ndvi_mae_base, 4),
        'Fine-Tuned NDVI-MAE': round(ndvi_mae_fine, 4),
        'NDVI-MAE Delta': round(ndvi_mae_fine - ndvi_mae_base, 4)
    })

df_metrics = pd.DataFrame(rows)

# Mean summary row
summary = {
    'Scene ID': 'MEAN',
    'Baseline PSNR (dB)': round(df_metrics['Baseline PSNR (dB)'].mean(), 2),
    'Fine-Tuned PSNR (dB)': round(df_metrics['Fine-Tuned PSNR (dB)'].mean(), 2),
    'PSNR Delta': round(df_metrics['PSNR Delta'].mean(), 2),
    'Baseline SSIM': round(df_metrics['Baseline SSIM'].mean(), 4),
    'Fine-Tuned SSIM': round(df_metrics['Fine-Tuned SSIM'].mean(), 4),
    'SSIM Delta': round(df_metrics['SSIM Delta'].mean(), 4),
    'Baseline NDVI-MAE': round(df_metrics['Baseline NDVI-MAE'].mean(), 4),
    'Fine-Tuned NDVI-MAE': round(df_metrics['Fine-Tuned NDVI-MAE'].mean(), 4),
    'NDVI-MAE Delta': round(df_metrics['NDVI-MAE Delta'].mean(), 4)
}
df_final = pd.concat([df_metrics, pd.DataFrame([summary])], ignore_index=True)

metrics_csv = os.path.join(RESULTS_DIR, 'sen2sr_lite_test_benchmark_metrics.csv')
df_final.to_csv(metrics_csv, index=False)
print('=' * 90)
print('GROUND-TRUTH BENCHMARK SANITY TABLE (NAIP Ground Truth vs. Baseline & Fine-Tuned)')
print('=' * 90)
print(df_final.to_markdown(index=False))
print(f'\nMetrics table saved to: {metrics_csv}')


## SECTION 2: Real-World Copernicus Satellite Scene Inference

> [!NOTE]
> **Visual Inspection Only — No Aerial Ground Truth Exists for this Scene**
> This section demonstrates full-scene sliding window tiled inference (with 2D Hann cosine window blending) on Copernicus Sentinel-2 captures. Edge gradient scores represent relative visual sharpness, not physical ground-truth accuracy.


In [ ]:
# Discover Copernicus ZIP file in Datasets
zip_candidates = glob.glob(os.path.join(DATASETS_DIR, '*.zip')) + glob.glob(os.path.join(DATASETS_DIR, '**/*.zip'), recursive=True)
assert len(zip_candidates) > 0, f"No ZIP archive found in {DATASETS_DIR}!"

TARGET_ZIP = zip_candidates[0]
print(f'Using Copernicus Sentinel-2 scene archive: {TARGET_ZIP}')

EXTRACT_DIR = '/content/copernicus_eval_lite'
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(TARGET_ZIP, 'r') as z:
    z.extractall(EXTRACT_DIR)

tifs = glob.glob(os.path.join(EXTRACT_DIR, '**/*.tif'), recursive=True) + glob.glob(os.path.join(EXTRACT_DIR, '**/*.jp2'), recursive=True)
print(f'Extracted {len(tifs)} raster files from scene archive.')


In [ ]:
BAND_NAMES = ['B04', 'B03', 'B02', 'B08'] # [Red, Green, Blue, NIR]

found = {}
for p in tifs:
    base = os.path.basename(p).upper()
    for b in BAND_NAMES:
        if re.search(rf'(^|[_.\-]){b}([_.\-]|$)', base):
            found[b] = p

print('Identified bands:', list(found.keys()))
assert len(found) >= 4, f"Missing required bands from {BAND_NAMES}, found: {list(found.keys())}"

channels = []
for b in BAND_NAMES:
    with rasterio.open(found[b]) as src:
        arr = src.read(1).astype(np.float32)
        if arr.max() > 255.0:
            arr = arr / SCALE_S2
        elif arr.max() > 2.0:
            arr = arr / 255.0
        channels.append(np.clip(arr, 0.0, 1.0))

s2_cube = np.stack(channels, axis=0) # (4, H, W) [R, G, B, N]
print(f'Loaded canonical Copernicus Sentinel-2 cube: shape={s2_cube.shape}, range=[{s2_cube.min():.4f}, {s2_cube.max():.4f}]')


In [ ]:
def run_sliding_window_hann(model, X, patch_size=128, stride=48, scale=4):
    ps_hr = patch_size * scale
    _, H, W = X.shape
    pad_h = max(0, patch_size - H)
    pad_w = max(0, patch_size - W)
    X_pad = F.pad(X, (0, pad_w, 0, pad_h), mode='replicate')
    Hp, Wp = X_pad.shape[-2:]

    # 2D Hann Cosine window for seamless patch blending
    w1d = torch.hann_window(ps_hr, periodic=False, dtype=torch.float32)
    w2d = (w1d.unsqueeze(1) * w1d.unsqueeze(0)).unsqueeze(0).unsqueeze(0).to(device)

    out_h, out_w = Hp * scale, Wp * scale
    acc = torch.zeros(1, X.shape[0], out_h, out_w, device=device)
    weight = torch.zeros(1, 1, out_h, out_w, device=device)

    y_steps = list(range(0, Hp - patch_size + 1, stride))
    if not y_steps or y_steps[-1] != Hp - patch_size:
        y_steps.append(Hp - patch_size)

    x_steps = list(range(0, Wp - patch_size + 1, stride))
    if not x_steps or x_steps[-1] != Wp - patch_size:
        x_steps.append(Wp - patch_size)

    with torch.no_grad():
        for y in y_steps:
            for x in x_steps:
                patch = X_pad[:, :, y:y+patch_size, x:x+patch_size].to(device)
                sr_p = model(patch)
                acc[:, :, y*scale:y*scale+ps_hr, x*scale:x*scale+ps_hr] += sr_p * w2d
                weight[:, :, y*scale:y*scale+ps_hr, x*scale:x*scale+ps_hr] += w2d

    blended = acc / torch.clamp(weight, min=1e-6)
    out = blended[:, :, :H*scale, :W*scale]
    return torch.clamp(out, 0.0, 1.0).squeeze(0).cpu().numpy()

s2_t = torch.from_numpy(s2_cube).unsqueeze(0)
print('Running full-scene Hann-window tiled inference...')
copernicus_base = run_sliding_window_hann(model_baseline, s2_t)
copernicus_fine = run_sliding_window_hann(model_finetuned, s2_t)
copernicus_bicubic = F.interpolate(s2_t, scale_factor=4, mode='bicubic', align_corners=False).squeeze(0).numpy()
print('Full-scene inference complete.')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

rgb_ref = copernicus_fine[[0, 1, 2]].transpose(1, 2, 0)
lo, hi = np.percentile(rgb_ref, [2, 98])

def stretch_cop(cube):
    rgb = cube[[0, 1, 2]].transpose(1, 2, 0)
    return np.clip((rgb - lo) / max(hi - lo, 1e-6), 0.0, 1.0)

axes[0].imshow(stretch_cop(copernicus_bicubic))
axes[0].set_title('1. Native 10m Input (Bicubic 4x)\nCopernicus Capture', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(stretch_cop(copernicus_base))
axes[1].set_title('2. Pretrained Baseline SEN2SR (2.5m)\nOfficial Weights', fontweight='bold')
axes[1].axis('off')

axes[2].imshow(stretch_cop(copernicus_fine))
axes[2].set_title('3. Fine-Tuned SEN2SR Lite (2.5m, Ours)\nShift-Tolerant Loss', fontweight='bold', color='green')
axes[2].axis('off')

plt.tight_layout()
cop_preview = os.path.join(RESULTS_DIR, 'sen2sr_lite_copernicus_fullscene.png')
plt.savefig(cop_preview, dpi=160, bbox_inches='tight')
plt.show()
print(f'Full scene preview saved to: {cop_preview}')
